# Chapter 3: Statistical Experiments and Significance Testing

**Author:** Ruben  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

This chapter covers experimental design and hypothesis testing:
- A/B testing for product experiments
- Hypothesis testing framework
- Permutation tests and resampling
- p-values and statistical significance
- t-tests, ANOVA, chi-square tests
- Multiple testing problems
- Power and sample size

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 3.1 A/B Testing

### Theory

**A/B Test** = controlled experiment with two groups:
- **Treatment (B)**: New version/feature
- **Control (A)**: Existing version/baseline

**Goal:** Determine if treatment causes measurable difference

**Key Requirements:**
1. **Random assignment**: Users randomly assigned to A or B
2. **Sufficient sample size**: Enough power to detect effect
3. **Single metric**: Define success metric upfront
4. **Simultaneous testing**: Run both at same time

**Why control group?**
- Accounts for external factors
- Establishes baseline
- Isolates treatment effect

In [ ]:
# A/B Test Example: Website conversion rates
np.random.seed(42)

# Control group A: existing page (5% conversion)
n_A = 1000
conversions_A = np.random.binomial(1, 0.05, n_A)
rate_A = conversions_A.mean()

# Treatment group B: new page (6% conversion - 20% improvement)
n_B = 1000
conversions_B = np.random.binomial(1, 0.06, n_B)
rate_B = conversions_B.mean()

print("A/B Test Results:")
print(f"Control (A):   {conversions_A.sum()}/{n_A} = {rate_A:.1%} conversion")
print(f"Treatment (B): {conversions_B.sum()}/{n_B} = {rate_B:.1%} conversion")
print(f"\nAbsolute difference: {(rate_B - rate_A):.1%}")
print(f"Relative lift: {((rate_B - rate_A) / rate_A):.1%}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
groups = ['Control (A)', 'Treatment (B)']
rates = [rate_A, rate_B]
colors = ['steelblue', 'coral']
axes[0].bar(groups, rates, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Conversion Rate')
axes[0].set_title('A/B Test: Conversion Rates')
axes[0].set_ylim([0, max(rates) * 1.2])
for i, rate in enumerate(rates):
    axes[0].text(i, rate + 0.002, f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')

# Stacked bar for outcomes
converted = [conversions_A.sum(), conversions_B.sum()]
not_converted = [n_A - conversions_A.sum(), n_B - conversions_B.sum()]
axes[1].bar(groups, converted, label='Converted', color='green', alpha=0.7, edgecolor='black')
axes[1].bar(groups, not_converted, bottom=converted, label='Not Converted', color='red', alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('A/B Test: Outcomes Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n❓ Question: Is this difference statistically significant or just random chance?")

## 3.2 Hypothesis Testing Framework

### Theory

**Hypothesis Test:** Statistical method to make decisions about population based on sample

**Null Hypothesis (H₀):**
- Default assumption: "no effect", "no difference"
- Example: "Treatment has no effect" (μ_A = μ_B)

**Alternative Hypothesis (H₁ or Hₐ):**
- What we want to show
- **One-sided**: μ_B > μ_A (treatment better)
- **Two-sided**: μ_B ≠ μ_A (treatment different)

**Test Logic:**
1. Assume H₀ is true
2. Calculate: "How unlikely is our observed data if H₀ true?"
3. If very unlikely → reject H₀

**Errors:**
- **Type I (α)**: False positive - reject true H₀
- **Type II (β)**: False negative - fail to reject false H₀
- **Power (1-β)**: Probability of detecting real effect

In [ ]:
# Hypothesis testing concept visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Type I and Type II errors
x = np.linspace(-4, 8, 1000)
null_dist = stats.norm.pdf(x, 0, 1)
alt_dist = stats.norm.pdf(x, 3, 1)

critical_value = 1.645  # One-sided test, α=0.05

axes[0].plot(x, null_dist, 'b-', linewidth=2, label='H₀: No effect')
axes[0].plot(x, alt_dist, 'r-', linewidth=2, label='H₁: True effect')
axes[0].fill_between(x, null_dist, where=(x >= critical_value), alpha=0.3, color='blue', label='Type I Error (α)')
axes[0].fill_between(x, alt_dist, where=(x <= critical_value), alpha=0.3, color='red', label='Type II Error (β)')
axes[0].axvline(critical_value, color='black', linestyle='--', linewidth=2)
axes[0].set_xlabel('Test Statistic')
axes[0].set_ylabel('Probability Density')
axes[0].set_title('Type I and Type II Errors')
axes[0].legend(fontsize=9)
axes[0].text(critical_value, 0.3, 'Critical\nValue', ha='center')

# Decision matrix
matrix_data = [
    ['Correct\nDecision', 'Type II Error\n(False Negative)'],
    ['Type I Error\n(False Positive)', 'Correct\nDecision']
]
axes[1].axis('tight')
axes[1].axis('off')
table = axes[1].table(cellText=matrix_data,
                      rowLabels=['Fail to Reject H₀', 'Reject H₀'],
                      colLabels=['H₀ True', 'H₀ False'],
                      cellLoc='center',
                      loc='center',
                      bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3)

# Color cells
for i in range(2):
    for j in range(2):
        cell = table[(i+1, j)]
        if (i == 0 and j == 0) or (i == 1 and j == 1):
            cell.set_facecolor('#90EE90')  # Light green for correct
        else:
            cell.set_facecolor('#FFB6C6')  # Light red for errors

axes[1].set_title('Decision Matrix', pad=20, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Common significance levels:")
print("α = 0.05 (5%): Standard in most fields")
print("α = 0.01 (1%): More conservative")
print("α = 0.10 (10%): More lenient (exploratory)")

## 3.3 Permutation Test (Resampling Approach)

### Theory

**Permutation Test** = non-parametric hypothesis test using resampling

**Logic:**
- **Under H₀**: Group labels are arbitrary (no real difference)
- **Test**: Randomly shuffle labels many times
- **Compare**: Observed difference vs shuffled differences

**Procedure:**
1. Compute observed test statistic (e.g., difference in means)
2. Pool all data together
3. Randomly reassign group labels
4. Compute statistic for permuted data
5. Repeat 1000s of times
6. p-value = proportion of permutations ≥ observed

**Advantages:**
- No distribution assumptions
- Works with any test statistic
- Intuitive interpretation